In [1]:
import mne
import os
import os.path as osp
import math
import pandas as pd
import numpy as np
import subprocess

import ctypes
import multiprocessing as mp

import sys
import wandb

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import time

from torch import nn
from torchsummary import summary

sys.path.insert(1, "../../")
sys.path.insert(1, "../")
from preprocess import Epoching, preprocess
from read_data import *
from utils import *
from models import deepConvNet, EEGSimpleConv

from numpy.random import seed
from braindecode.models import Deep4Net
from braindecode.training.losses import CroppedLoss

import torch

from multiprocessing import Pool
import random

seed(2002012)  # set seed for reproducibility
torch.manual_seed(2002012)

random.seed(2002012)
seed(2002012)  # set seed for reproducibility
torch.manual_seed(2002012)
torch.cuda.manual_seed(2002012)
torch.cuda.manual_seed_all(2002012)


# torch.use_deterministic_algorithms()
# torch.use_deterministic_algorithms(True)
# torch.backends.cudnn.benchmark = False
# torch.backends.cudnn.deterministic = True
# def saliency_map(model, valid_loader, criterion, device):
#     model.eval()
#     total_loss = 0
#     correct = 0
#     saliency = torch.zeros((n_chans,input_window_samples))
#     with torch.no_grad():
#         for batch in valid_loader:
#             data = batch[0]
#             data = data.to(device)
#             data.requires_grad = True
#             
#             target = batch[1]
# 
#             output = model(data )
#             output = torch.sum(output, dim=0)/ output.shape[0]
#        
#             output_right = output[1]
#             output_right.backward(retain_graph=False)
#             saliency += data.grad.data.abs()[0]
#             
# 
#     saliency = saliency /len(valid_loader.dataset)
#     saliency = saliency.cpu().numpy()
#     
#     return saliency
# 
# saliency = saliency_map(model, test_data, criterion, device)
# 

C:\Users\dtrocell\AppData\Local\anaconda3\envs\ml_deep_bci\lib\site-packages\torchaudio\backend\utils.py:62: UserWarning: No audio backend is available.
  warnings.warn("No audio backend is available.")


In [2]:

def saliency_map(model, valid_loader, device, class_index=1):
    model.eval()
    saliency = torch.zeros((n_chans, input_window_samples), device=device)

    for batch in valid_loader:
        data, target = batch
        mask = target == class_index
        data = data[mask]

        data = data.to(device)
        data.requires_grad = True

        output = model(data)
        output = torch.sum(output, dim=0) / output.shape[0]

        #right
        output_right = output[class_index]
        # If output is not a scalar, consider using torch.sum(output).backward()
        output_right.backward()
        # Assuming data.grad is not None and has the same shape as data
        if data.grad is not None:
            saliency += data.grad.abs().sum(dim=0)  # Sum over the batch
        else:
            raise ValueError("data.grad is None")

    saliency = saliency / len(valid_loader.dataset)
    # saliency_left = saliency_left / len(valid_loader.dataset)

    saliency = saliency.cpu().numpy()
    # saliency_left = saliency_left.cpu().numpy()
    # return saliency_left , saliency

    return saliency

In [3]:


# model = "deep_convnet"
n_classes = 2
n_chans = 27
sfreq = 512
input_window_samples = int(sfreq * 4)

name_model = "Deep4Net"
name_folder = "Deep4Net"
n_epochs = 150
batch_size = 256
lr = 0.001
dic_models = {}
path_model = f"model/{name_folder}/"
# f"model/{config.model}/{config.model}_{config.test_subject}_{save_name}.pt"
save_name = f"{name_model}_{n_epochs}_epochs_{batch_size}_batch_size_{lr}_lr"

accuracy = pd.read_csv("Results/perf_benchmark.csv")

accuracy = accuracy.set_index("subject")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # this specify that we want the first GPU
for index in accuracy.index:
    dic_models[index] = torch.load(path_model + f"{name_model}_{index}_{save_name}.pt", map_location=device)

In [4]:


init_path = "../../../../Dataset"
dict_config = {
    'model':'EEGSimpleConv',
    'params': [125,2,75,15],
    'dataset': 'Large',
    'runs': 1,
    'path': 'Dataset/',
    'evaluation': 'cross',
    'EA': False,
    'cross': True,
    'ft': False,
    'reg_subject': False,
    'preload_reg': False,
    'n_epochs': 50,
    'mixup': False,
    'use_wandb': True,
    'load_model': False,
    'save_model': False}


subject_test = 0 

X_EA , X_online, Y = load_data_yassine(dict_config, path=init_path)

_, _, _, test_data = loaders_cross_yassine(subject_test, X_EA, X_online, Y, dataset="Large", reg_subject=dict_config["reg_subject"])


FileNotFoundError: [Errno 2] No such file or directory: '../../../../Dataset/Large/X_s.pt'

In [ ]:

def lastConvLengthDeep4Net(input_size_x):
    return math.floor((input_size_x - 9) / 3)

final_length = lastConvLengthDeep4Net(lastConvLengthDeep4Net(lastConvLengthDeep4Net(
        lastConvLengthDeep4Net(input_window_samples))))  # Made my alex Pepy manage the final length if input window is too small
model = Deep4Net(
    in_chans=n_chans,
    n_classes=n_classes,
    input_window_samples=input_window_samples,
    final_conv_length=final_length )

model.load_state_dict(dic_models[subject_test])
# put the model on the GPU
model = model.to(device)

summary(model, (n_chans, input_window_samples))
saliency = saliency_map(model, test_data, device, class_index=1)
# plot the saliency map in temportal domain
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
saliency_temporal = saliency.mean(axis=0)
saliency_temporal = saliency_temporal / saliency_temporal.sum() * 100
# reduce the time resolution by 10 times by sliding window


plt.plot(saliency_temporal)
plt.xlabel("Time [samples]")
plt.ylabel("Saliency in %")

In [ ]:
plt.plot(saliency_temporal.cumsum())
plt.xlabel("Time [samples]")
plt.ylabel("Cumulative explanation in %")

In [ ]:
if len(saliency_temporal) % 50 != 0:
    saliency_temporal_temp = saliency_temporal[:-(len(saliency_temporal) % 50)]

saliency_temporal_10 = saliency_temporal_temp.reshape(-1, 50).sum(axis=1)

# Plot the resulting array
plt.figure(figsize=(10, 5))
plt.plot(saliency_temporal_10, )
plt.xlabel("Time [samples]")
plt.ylabel("Saliency in %")
plt.show()

In [ ]:
if len(saliency_temporal) % 20 != 0:
    saliency_temporal_temp = saliency_temporal[:-(len(saliency_temporal) % 20)]

saliency_temporal_10 = saliency_temporal_temp.reshape(-1, 20).sum(axis=1)

# Plot the resulting array
plt.figure(figsize=(10, 5))
plt.plot(saliency_temporal_10, )
plt.xlabel("Time [samples]")
plt.ylabel("Saliency in %")
plt.show()

In [ ]:
plt.bar(range(len(saliency_temporal_10)), saliency_temporal_10)

In [ ]:

# plot the saliency map in spatial domain
import mne

# load the channel location
montage = mne.channels.make_standard_montage("standard_1020")
ch_names = ['Fz', 'FCz', 'Cz', 'CPz', 'Pz', 'C1', 'C3', 'C5', 'C2', 'C4', 'C6', 'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4',
            'CP6', 'P4', 'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3']
ch_pos = montage.get_positions()["ch_pos"]
ch_pos = np.array([ch_pos[ch_name][:2] for ch_name in ch_names])

ch_pos = montage.get_positions()["ch_pos"]
ch_pos = np.array([ch_pos[ch_name][:2] for ch_name in ch_names])


In [ ]:
import seaborn as sns

sns.heatmap(saliency, cmap="Reds", xticklabels=50, yticklabels=ch_names)


In [ ]:

# plot the saliency map in spatial domain
fig, ax = plt.subplots(figsize=(10, 10))
saliency_spatial = saliency.mean(axis=1)
saliency_spatial = saliency_spatial / saliency_spatial.sum() * 100
im , _  = mne.viz.plot_topomap(saliency_spatial, ch_pos, names=ch_names, cmap="Reds", axes=ax, show=False)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Saliency in %", rotation=270, labelpad=30)

plt.show()

#  print the saliency map in spatial domain for each time point
it does not work

In [ ]:

n = 8  # number of time points to average

if saliency.shape[1] % n != 0:
    saliency_n = saliency[:, : - (saliency.shape[1] % n)]

saliency_n = saliency_n.reshape(saliency.shape[0], -1,  n)
saliency_n = saliency_n / saliency_n.sum() * 100



In [ ]:
saliency_n = np.array([saliency[: , i::n ] for i in range(n)])  

In [ ]:
saliency_spatial_n = saliency_n.sum(axis=2)
saliency_spatial_n = saliency_spatial_n / saliency_spatial_n.sum() * 100

In [ ]:
# print the saliency map in spatial domain

# fig, ax = plt.subplots(, figsize=(10, 10))
fig, ax = plt.subplots(1, saliency_spatial_n.shape[0], figsize=(40, 10))

for i in range(saliency_spatial_n.shape[0]) : 
    im , _  = mne.viz.plot_topomap(saliency_spatial_n[i,:], ch_pos, names=ch_names, cmap="Reds", axes=ax[i], show=False)
    # plt.title(f"Time point {i*n} to {(i+1)*n}")
    ax[i].set_title(f"Time point {i*n} to {(i+1)*n}")
plt.show()

# saliency map for all 87 subjects

In [ ]:
saliency_temporal_per_subject = np.zeros((87, saliency_temporal.shape[0]))
saliency_spatial_per_subject = np.zeros((87, saliency_spatial.shape[0]))
for test_subject in range(87):


    _, _, _, test_data = loaders_cross_yassine(test_subject, X_EA, X_online, Y, dataset="Large",
                                               reg_subject=dict_config["reg_subject"])
    
    model = Deep4Net(
        in_chans=n_chans,
        n_classes=n_classes,
        input_window_samples=input_window_samples,
        final_conv_length=final_length)
    
    model.load_state_dict(dic_models[test_subject])
    # put the model on the GPU
    model = model.to(device)
    
    saliency = saliency_map(model, test_data, device, class_index=1)
    # plot the saliency map in temportal domain
    
    saliency_temporal = saliency.mean(axis=0)
    saliency_temporal = saliency_temporal / saliency_temporal.sum() * 100
    saliency_temporal_per_subject[test_subject] = saliency_temporal
    
    
    saliency_spatial = saliency.mean(axis=1)
    saliency_spatial = saliency_spatial / saliency_spatial.sum() * 100
    saliency_spatial_per_subject[test_subject] = saliency_spatial
    


In [ ]:
np.save( "Results/saliency_temporal_per_subject_05_45.npy", saliency_temporal_per_subject)
np.save("Results/saliency_spatial_per_subject_05_45.npy", saliency_spatial_per_subject)

In [ ]:
saliency_temporal_per_subject = np.load("Results/saliency_temporal_per_subject_05_45.npy")
saliency_spatial_per_subject = np.load("Results/saliency_spatial_per_subject_05_45.npy")

saliency_temporal_per_subject

In [ ]:
plt.figure(figsize=(10, 5))


for i in range(10):
    plt.plot(saliency_temporal_per_subject[i], alpha=0.1, c="grey")
plt.plot(saliency_temporal_per_subject.mean(axis=0),c= "black")
plt.xlabel("Time [samples]")
plt.ylabel("Saliency in %")


# + std et - std 
